# Drinfeld modular polynomials

A guided tour of the code and data accompanying

> F. Breuer and F. Pazuki, *Divisibility of the coefficients of Drinfeld modular polynomials*.

Let $A = \mathbb{F}_q[T]$ and let $N \in A$ be monic. The Drinfeld modular polynomial $\Phi_N(X,Y) \in A[X,Y]$ vanishes exactly at the pairs of $j$-invariants of rank 2 Drinfeld $A$-modules linked by a cyclic isogeny of degree $N$. It is symmetric, monic, and of degree $\psi(N) = |N|\prod_{P \mid N}(1 + |P|^{-1})$ in each variable. The paper shows that its coefficients are highly divisible by primes of small degree, and this notebook lets you look at that.

**What this notebook does.** Section 1 computes one modular polynomial from scratch (a second or two). Everything after that works with the certified polynomials stored in `data/`, so nothing expensive is recomputed and the whole notebook runs in about a minute.

**What it does not do.** It is an illustration, not the verification: `sage paper/verify_paper.py` checks every numerical claim of the paper, and `sage paper/recompute.py all` recomputes all the polynomials (about six minutes).

**To run it:** open it with the SageMath Jupyter kernel from the root directory of the repository.

In [1]:
from drinfeld_modpoly import (drinfeld_modular_polynomial, drinfeld_modular_polynomial_at_zero,
                              psi, height, height_bound, monic_primes, valuation,
                              ReducedDrinfeldModule)
from drinfeld_modpoly.analysis import (coefficient_dict, shifted, specialization, C_J, n_opt,
                                       bound_holds, kronecker_congruence,
                                       newton_root_valuations, is_monic, is_symmetric)
from drinfeld_modpoly.special import (sqrtT_singular_modulus, conductor_singular_modulus,
                                      bassa_beelen_phi_T)
from drinfeld_modpoly.database import load_level, load_all
from drinfeld_modpoly.display import CoefficientTable, Table, factored

## 1. Computing a modular polynomial

`drinfeld_modular_polynomial(N)` implements Algorithm 1 of the paper. It fixes a prime $P \nmid N$ and works in $E[\pi]/(\pi^K)$, where $E$ is a finite field and $\pi$ stands for $P$: it deforms the Drinfeld module $\phi^0 : T \mapsto T + \tau^2$ with $j = 0$ along $\psi(N)+1$ directions, lifts the $N$-torsion, forms $\Phi_N(X, y_s) = \prod_C (X - j(\tilde\varphi^s/C))$ over the cyclic kernels $C$, interpolates in $Y$, and reads the coefficients off their $P$-adic expansions.

The precision $K$ comes from the proven height bound of Breuer–Pazuki–Ran, so the output is the exact $\Phi_N$ (Proposition 5.1).

Here is $\Phi_{T^2+T+1}$ for $q = 2$.

In [2]:
A.<T> = GF(2)[]
N = T^2 + T + 1
Phi, cert = drinfeld_modular_polynomial(N, certificate=True)

Table([[cert["place"], cert["height_bound"], cert["precision"], "F_2^%d" % cert["extension_degree"]]],
      header=["base place P", "height bound h", "precision K", "working field E"])

base place P,height bound h,precision K,working field E
T,102,129,F_2^6


In [3]:
X, Y = Phi.parent().gens()
print("degree in X: %d = psi(N) = %d" % (Phi.degree(X), psi(N)))
print("height: %d  (proven bound: %d)" % (height(Phi), height_bound(N)))
print("symmetric:", Phi(X, Y) == Phi(Y, X))

degree in X: 5 = psi(N) = 5
height: 24  (proven bound: 102)
symmetric: True


### Table 1 of the paper

Write $\Phi_N = \sum_{m,n} a_{m,n} X^m Y^n$. **Theorem 1.2(1)** says that for every prime $P \nmid N$ of degree one,

$$v_P(a_{m,n}) \;\ge\; (q+1)\,\big(\psi(N) - m - n\big) \qquad (m + n < \psi(N)).$$

Here $q + 1 = 3$, $\psi(N) = 5$, and both primes of degree one qualify, so $a_{m,n}$ is divisible by $\big(T(T+1)\big)^{3(5-m-n)}$. In the table below this predicted factor is in bold, and what remains is factored next to it. (Only $a_{m,n}$ with $m \ge n$ are listed, since $a_{m,n} = a_{n,m}$.)

In [4]:
predicted = {P: (psi(N), 3) for P in (T, T + 1)}     # v_P(a_mn) >= 3 (psi(N) - m - n)
CoefficientTable(Phi, bounds=predicted)

a_(1,1) = [T^9 * (T + 1)^9] * T^2 * (T + 1)^2
a_(2,0) = [T^9 * (T + 1)^9] * (T^2 + T + 1)^3
a_(2,1) = [T^6 * (T + 1)^6] * (T^2 + T + 1) * (T^4 + T + 1)
a_(2,2) = [T^3 * (T + 1)^3] * T^2 * (T + 1)^2 * (T^2 + T + 1) * (T^4 + T + 1)
a_(3,0) = [T^6 * (T + 1)^6] * (T^2 + T + 1)^2
a_(3,1) = [T^3 * (T + 1)^3] * (T^2 + T + 1)
a_(3,2) = T * (T + 1) * (T^2 + T + 1) * (T^3 + T + 1) * (T^3 + T^2 + 1)
a_(3,3) = T^2 * (T + 1)^2 * (T^2 + T + 1)
a_(4,0) = [T^3 * (T + 1)^3] * (T^2 + T + 1)
a_(4,1) = T^2 + T + 1
a_(4,2) = T^2 + T + 1
a_(4,4) = 1
a_(5,0) = 1
zero: a_(0,0), a_(1,0), a_(4,3), a_(5,1), a_(5,2), a_(5,3), a_(5,4), a_(5,5)

The bound is attained exactly, for instance at $a_{2,0}$. The two vanishing coefficients $a_{0,0}, a_{1,0}$ reflect $C_0(N,0) = \mathrm{ord}_X \Phi_N(X,0) = 2$: two of the five cyclic kernels are stable under $\mathrm{End}(\phi^0) = \mathbb{F}_4[T]$.

### The result does not depend on the choices made

The base place, the root of $P$ and the deformation parameters are arbitrary. Recomputing at two other places changes every intermediate object and gives the same polynomial, which is also the one stored in `data/`.

In [5]:
for P in (T + 1, T^3 + T + 1):
    assert drinfeld_modular_polynomial(N, place=P) == Phi
assert Phi == load_level(N)[0]
print("identical at the places T, T + 1 and T^3 + T + 1, and identical to data/")

identical at the places T, T + 1 and T^3 + T + 1, and identical to data/


## 2. The data set

`data/phi/` contains $\Phi_N$ for the 20 levels of the paper. Each is monic and symmetric, satisfies the Kronecker congruence $\Phi_N \equiv (X^{|N|} - Y)(X - Y^{|N|}) \bmod N$ when $N$ is prime, and has height (the largest $T$-degree of a coefficient) below the proven bound. The last two columns record how each polynomial was computed.

In [6]:
data = load_all()                      # triples (N, Phi_N, record), read from data/phi
rows = []
for N, Phi, rec in data:
    q = N.base_ring().cardinality()
    assert is_monic(Phi) and is_symmetric(Phi)
    kronecker = "-"
    if N.is_irreducible():
        assert kronecker_congruence(Phi, N)
        kronecker = "holds"
    h, hb = height(Phi), height_bound(N)
    assert h <= hb
    comp = rec["computation"]
    rows.append([q, N, psi(N), h, hb, kronecker, comp["place"], "%.1f s" % comp["seconds"]])
Table(rows, header=["q", "N", "psi(N)", "height", "proven bound", "Kronecker", "computed at", "time"])

q,N,psi(N),height,proven bound,Kronecker,computed at,time
2,T,3,12,57,holds,T + 1,1.8 s
2,T + 1,3,12,57,holds,T,0.1 s
2,T^2,6,24,121,-,T + 1,3.9 s
2,T^2 + T,9,24,177,-,T^2 + T + 1,3.1 s
2,T^2 + T + 1,5,24,102,holds,T,1.1 s
2,T^3,12,84,258,-,T + 1,22.4 s
2,T^3 + T + 1,9,72,200,holds,T,19.6 s
3,T,4,36,149,holds,T + 1,0.2 s
3,T + 1,4,36,149,holds,T + 2,0.2 s
3,T + 2,4,36,149,holds,T,0.2 s


## 3. Primes of degree one: Theorem 1.2(1) is sharp

For a prime $P \nmid N$ let $C_0(N,P) = \mathrm{ord}_X\big(\Phi_N(X,0) \bmod P\big)$, and let

$$n_{\mathrm{opt}}(N,P) = \min\Big\{ \frac{v_P(a_{m,n})}{C_0(N,P) - m - n} \;:\; m + n < C_0(N,P),\ a_{m,n} \ne 0 \Big\}$$

be the best constant that could replace $q+1$ in the theorem. For $\deg P = 1$ one has $C_0(N,P) = \psi(N)$ (Proposition 2.3), and the theorem says $n_{\mathrm{opt}} \ge q + 1$. In the data this is an equality at every one of the 44 pairs $(N, P)$.

In [7]:
rows, pairs = [], 0
for N, Phi, _ in data:
    q = N.base_ring().cardinality()
    primes = [P for P in monic_primes(N.parent(), 1) if not P.divides(N)]
    values = set()
    for P in primes:
        assert C_J(Phi, 0, 0, P) == psi(N)                # Proposition 2.3
        assert bound_holds(Phi, P, psi(N), q + 1)         # Theorem 1.2(1)
        values.add(n_opt(Phi, P, psi(N))[0])
        pairs += 1
    assert values <= {q + 1}
    rows.append([q, N, psi(N), len(primes), ", ".join(str(v) for v in values) or "-"])
print(pairs, "pairs (N, P)")
Table(rows, header=["q", "N", "psi(N)", "primes P of degree 1 not dividing N", "n_opt(N, P)"])

44 pairs (N, P)


q,N,psi(N),primes P of degree 1 not dividing N,"n_opt(N, P)"
2,T,3,1,3
2,T + 1,3,1,3
2,T^2,6,1,3
2,T^2 + T,9,0,-
2,T^2 + T + 1,5,2,3
2,T^3,12,1,3
2,T^3 + T + 1,9,2,3
3,T,4,2,4
3,T + 1,4,2,4
3,T + 2,4,2,4


### Where the minimum is attained

The lowest coefficient is $a_{c,0} = \pm\prod_C j(\phi^0/C)$, over the kernels with $j(\phi^0/C) \ne 0$, and $v_P\big(j(\phi^0/C)\big) = (q+1)\big(1 + v_P(S_C)\big)$ with $S_C = \sum_{0 \ne x \in C} x^{1-q}$. So $a_{c,0}$ attains the bound exactly when no kernel has $P \mid S_C$. That happens in 42 of the 44 cases. One of the two exceptions is $(q, N, P) = (2,\ T^3+T+1,\ T+1)$:

In [8]:
A.<T> = GF(2)[]
N, P = T^3 + T + 1, T + 1
Phi = load_level(N)[0]
print("v_P(a_00) = %d, while (q+1) psi(N) = %d" % (valuation(Phi.constant_coefficient(), P), 3 * psi(N)))
n, where = n_opt(Phi, P, psi(N))
print("n_opt =", n, "is attained at (m, n) =", where)

v_P(a_00) = 36, while (q+1) psi(N) = 27
n_opt = 3 is attained at (m, n) = [(0, 3), (0, 5), (0, 7), (1, 2), (1, 4), (1, 6), (2, 1), (3, 0), (4, 1), (5, 0), (6, 1), (7, 0)]


`ReducedDrinfeldModule(N, P)` computes in characteristic $P$: the $N$-torsion of the reduction of $\phi^0$, its $\psi(N)$ cyclic kernels, their kernel polynomials and the residues $\bar S_C$. This involves no height bound at all, and it explains the valuation $36$: three kernels have $P \mid S_C$, and the Newton polygon of $\Phi_N(X,0)$ at $P$ shows three roots of valuation $6$ instead of $3$.

In [9]:
M = ReducedDrinfeldModule(N, P)
S = [M.S(C) for C in M.kernels]
print("%d cyclic kernels, %d of them with S_C = 0 mod P" % (len(M.kernels), S.count(0)))
print("root valuations of Phi_N(X, 0) at P:", newton_root_valuations(specialization(Phi, 0, 0), P))

9 cyclic kernels, 3 of them with S_C = 0 mod P
root valuations of Phi_N(X, 0) at P: {6: 3, 3: 6}


## 4. Primes of higher degree

At a prime of **even** degree the reduction of $j = 0$ is ordinary, and $C_0(N,P) = C_0(N,0)$: no divisibility. At a prime of **odd** degree it is supersingular, and **Theorem 1.2(2)** gives

$$v_P(a_{m,n}) \;\ge\; (q+1)\,\big(C_0(N,P) - m - n\big),$$

which says something new only when $C_0(N,P) > C_0(N,0)$. To see how often such *excess* occurs, `data/phi_at_zero/` contains $\Phi_N(X,0)$ for all 24 monic $N$ of degree 3 and 4 over $\mathbb{F}_2$. The table lists $C_0(N,P)$ at the two primes of degree 3.

In [10]:
A.<T> = GF(2)[]
P31, P32 = T^3 + T + 1, T^3 + T^2 + 1

def order(f, P=None):
    # ord_X of f, modulo P if P is given
    return next(i for i, c in enumerate(f.list()) if (c if P is None else c % P) != 0)

rows, with_excess = [], 0
for N, f, _ in load_all(at_zero=True):
    c0 = order(f)
    cells, excess = [], False
    for P in (P31, P32):
        if P.divides(N):
            cells.append("(P divides N)")
            continue
        C = order(f, P)
        assert C >= c0 and (C - c0) % 3 == 0        # the excess is a multiple of q + 1
        excess = excess or C > c0
        cells.append("%d%s" % (C, "   <-- excess" if C > c0 else ""))
    with_excess += excess
    rows.append([factored(N), psi(N), c0] + cells)
print("excess at a prime of degree 3 for %d of the %d levels" % (with_excess, len(rows)))
Table(rows, header=["N", "psi(N)", "C_0(N,0)", "C_0(N, T^3+T+1)", "C_0(N, T^3+T^2+1)"])

excess at a prime of degree 3 for 20 of the 24 levels


N,psi(N),"C_0(N,0)","C_0(N, T^3+T+1)","C_0(N, T^3+T^2+1)"
T^3,12,0,0,3 <-- excess
(T + 1) * (T^2 + T + 1),15,0,0,3 <-- excess
T * (T + 1)^2,18,0,3 <-- excess,6 <-- excess
T^3 + T + 1,9,0,(P divides N),0
T^2 * (T + 1),18,0,6 <-- excess,3 <-- excess
T^3 + T^2 + 1,9,0,0,(P divides N)
T * (T^2 + T + 1),15,0,3 <-- excess,0
(T + 1)^3,12,0,3 <-- excess,0
T^4,24,0,6 <-- excess,0
(T + 1)^4,24,0,0,6 <-- excess


The excess is always a multiple of $q + 1 = 3$, because $\mathrm{Aut}(\phi^0) = \mathbb{F}_{q^2}^*$ permutes the kernels it counts in orbits of length $q+1$.

For $N = T^3$ the full polynomial is in the data set, so Theorem 1.2(2) can be seen at work at the excess prime $P = T^3 + T^2 + 1$, where $C_0(N,P) = 3$. The count of kernels in characteristic $P$ confirms this value independently, and the bound is attained.

In [11]:
N, P = T^3, T^3 + T^2 + 1
Phi = load_level(N)[0]
C = C_J(Phi, 0, 0, P)
print("C_0(N, P) = %d from the polynomial, %d from counting kernels in characteristic P"
      % (C, ReducedDrinfeldModule(N, P).count_isogenous(0)))
print("n_opt =", n_opt(Phi, P, C)[0])
Table([[(m, n), valuation(a, P), 3 * (C - m - n)]
       for (m, n), a in sorted(coefficient_dict(Phi).items()) if m + n < C],
      header=["(m, n)", "v_P(a_mn)", "(q+1)(C_0(N,P) - m - n)"])

C_0(N, P) = 3 from the polynomial, 3 from counting kernels in characteristic P
n_opt = 3


"(m, n)",v_P(a_mn),"(q+1)(C_0(N,P) - m - n)"
"(0, 0)",9,9
"(0, 1)",6,6
"(0, 2)",3,3
"(1, 0)",6,6
"(1, 1)",3,3
"(2, 0)",3,3


## 5. Ramified $j$-invariants and fractional exponents

Now shift the variables: write $\Phi_N(X + J, Y + J) = \sum a_{m,n}X^mY^n$ for a $j$-invariant $J \in A$. **Theorem 4.1** says that if $0 < v := v_T(J) < q+1$ and $T \nmid N$, then

$$v_T(a_{m,n}) \;\ge\; \Big(v + \frac1e\Big)\big(\psi(N) - m - n\big), \qquad e = \frac{q+1}{\gcd(q+1,\,v)}.$$

No complex multiplication is needed. The constant is fractional, and it exceeds by $\frac1e$ what Theorem 1.2 alone would give.

For $q = 3$ and $N = T + 1$ (so $\psi(N) = 4$), the table shows, for each $k = m + n$, the smallest valuation among the $a_{m,n}$ and the bound. The last two rows are singular moduli: $J_s$ has CM by $\mathbb{F}_q[\sqrt T]$ and $J_c$ has CM by the order of conductor $T$ in $\mathbb{F}_{q^2}[T]$.

In [12]:
A.<T> = GF(3)[]
Js, Jc = sqrtT_singular_modulus(A), conductor_singular_modulus(A)
N = T + 1
Phi = load_level(N)[0]
rows = []
for J in (T, T^2, T^3, Js, Jc):
    v = valuation(J, T)
    e = 4 // gcd(4, v)
    c = v + 1/e
    S = coefficient_dict(shifted(Phi, J))
    cells = []
    for k in range(4):
        least = min(valuation(a, T) for (m, n), a in S.items() if m + n == k)
        assert least >= c * (4 - k)
        cells.append("%s  (bound %s)" % (least, c * (4 - k)))
    rows.append([factored(J), v, e, c] + cells)
Table(rows, header=["J", "v", "e", "v + 1/e"] + ["m + n = %d" % k for k in range(4)])

J,v,e,v + 1/e,m + n = 0,m + n = 1,m + n = 2,m + n = 3
T,1,4,5/4,5 (bound 5),4 (bound 15/4),3 (bound 5/2),2 (bound 5/4)
T^2,2,2,5/2,10 (bound 10),8 (bound 15/2),5 (bound 5),3 (bound 5/2)
T^3,3,4,13/4,13 (bound 13),17 (bound 39/4),8 (bound 13/2),4 (bound 13/4)
T^2 * (T + 1)^4,2,2,5/2,10 (bound 10),8 (bound 15/2),5 (bound 5),3 (bound 5/2)
2 * T * (T + 1)^4 * (T + 2)^4,1,4,5/4,5 (bound 5),4 (bound 15/4),3 (bound 5/2),2 (bound 5/4)


Across the whole data set the constant $v + \frac1e$ is attained for every level, in all 36 pairs $(J, N)$ with $J$ not a singular modulus:

In [13]:
rows, pairs = [], 0
for q in (2, 3):
    A = GF(q)['T']
    T = A.gen()
    levels = [N for N, _, _ in data if N.parent() is A and not T.divides(N)]
    for J in ([T, T^2, T*(1 + T), T + T^4] if q == 2 else [T, T^2, T^3]):
        v = valuation(J, T)
        e = (q + 1) // gcd(q + 1, v)
        c = v + 1/e
        found = set()
        for N in levels:
            Phi = load_level(N)[0]
            S = shifted(Phi, J)
            assert C_J(Phi, J, J, T) == psi(N) and bound_holds(S, T, psi(N), c)   # Theorem 4.1
            found.add(n_opt(S, T, psi(N))[0])
            pairs += 1
        assert found == {c}
        rows.append([q, J, v, e, c, len(levels), ", ".join(str(x) for x in found)])
print(pairs, "pairs (J, N)")
Table(rows, header=["q", "J", "v = v_T(J)", "e", "v + 1/e", "levels N", "n_opt at T"])

36 pairs (J, N)


q,J,v = v_T(J),e,v + 1/e,levels N,n_opt at T
2,T,1,3,4/3,3,4/3
2,T^2,2,3,7/3,3,7/3
2,T^2 + T,1,3,4/3,3,4/3
2,T^4 + T,1,3,4/3,3,4/3
3,T,1,4,5/4,8,5/4
3,T^2,2,2,5/2,8,5/2
3,T^3,3,4,13/4,8,13/4


### The singular modulus $J_s$: one table for nine levels

For $q = 3$, $J_s = T^2(1+T)^4$. The behaviour at a prime $P$ depends only on how $P$ splits in $k(\sqrt T)$ — and, at primes of degree 2, on whether $J_s$ reduces to the unique supersingular invariant $j_P$. The cell below computes $C_J(N,P)$ and $n_{\mathrm{opt}}$ for all nine levels $N \ne T$ of the data set and checks that the outcome is the same for every $N$; this reproduces the table of Section 5.5.

In [14]:
A.<T> = GF(3)[]
Js = sqrtT_singular_modulus(A)
primes = monic_primes(A, 1) + monic_primes(A, 2)
outcome = {P: set() for P in primes}
levels = [N for N, _, _ in data if N.parent() is A and N != T]
for N in levels:
    Phi = load_level(N)[0]
    S = shifted(Phi, Js)
    generic = C_J(Phi, Js)                                   # C_J(N, 0)
    for P in primes:
        if not P.divides(N):
            C = C_J(Phi, Js, Js, P)
            assert C in (psi(N), generic)
            outcome[P].add(("psi(N)" if C == psi(N) else "C_J(N,0)", n_opt(S, P, C)[0]))
rows = []
for P in primes:
    (C, n), = outcome[P]                                     # the same for all levels
    rows.append([P, valuation(Js, P), C, "none (ordinary)" if n is None else n])
print(len(levels), "levels")
Table(rows, header=["P", "v_P(J_s)", "C_J(N, P)", "n_opt"])

9 levels


P,v_P(J_s),"C_J(N, P)",n_opt
T,2,psi(N),5/2
T + 1,4,psi(N),4
T + 2,0,"C_J(N,0)",none (ordinary)
T^2 + 1,0,"C_J(N,0)",none (ordinary)
T^2 + T + 2,0,psi(N),1
T^2 + 2*T + 2,0,psi(N),1


At $P = T$ (ramified) the constant is $\frac{q+2}{2} = \frac52$ from Theorem 4.1; at $P = T+1$ (inert) it is $q + 1 = 4$; at $T^2+T+2$ and $T^2+2T+2$, where $J_s \equiv j_P$, it is $1$ (Proposition 2.4); at the other primes the reduction is ordinary.

### The conductor-$T$ singular modulus $J_c$

**Proposition 4.5:** $\Phi_T(X, 0) = (X - J_c)^{q+1}$ with $J_c = (-1)^q\,T\,(T^{q-1}-1)^{q+1}$. This holds for all five values of $q$ in the data set.

In [15]:
for N, Phi, _ in data:
    A = N.parent()
    T = A.gen()
    if N == T:
        q = A.base_ring().cardinality()
        Jc = conductor_singular_modulus(A)
        x = A['x'].gen()
        assert specialization(Phi, 0, 0) == ((x - Jc)^(q + 1)).list()
        print("q = %d:  Phi_T(X, 0) = (X - J_c)^%d  with  J_c = %s" % (q, q + 1, factored(Jc)))

q = 2:  Phi_T(X, 0) = (X - J_c)^3  with  J_c = T * (T + 1)^3
q = 3:  Phi_T(X, 0) = (X - J_c)^4  with  J_c = 2 * T * (T + 1)^4 * (T + 2)^4
q = 4:  Phi_T(X, 0) = (X - J_c)^5  with  J_c = T * (T + 1)^5 * (T + y)^5 * (T + y + 1)^5
q = 5:  Phi_T(X, 0) = (X - J_c)^6  with  J_c = 4 * T * (T + 1)^6 * (T + 2)^6 * (T + 3)^6 * (T + 4)^6
q = 9:  Phi_T(X, 0) = (X - J_c)^10  with  J_c = 2 * T * (T + 1)^10 * (T + 2)^10 * (T + 2*y)^10 * (T + 2*y + 1)^10 * (T + 2*y + 2)^10 * (T + y)^10 * (T + y + 1)^10 * (T + y + 2)^10


Since $v_T(J_c) = 1$, Theorem 4.1 gives the constant $\frac{q+2}{q+1}$ at $T$ (**Corollary 4.6**), while at the other primes of degree one $v_P(J_c) = q+1$ and the constant is $q+1$. Both are attained for every level:

In [16]:
rows = []
for q in (2, 3):
    A = GF(q)['T']
    T = A.gen()
    Jc = conductor_singular_modulus(A)
    found = {}
    levels = [N for N, _, _ in data if N.parent() is A and not T.divides(N)]
    for N in levels:
        Phi = load_level(N)[0]
        S = shifted(Phi, Jc)
        for P in monic_primes(A, 1):
            if not P.divides(N):
                assert C_J(Phi, Jc, Jc, P) == psi(N)
                found.setdefault(P, set()).add(n_opt(S, P, psi(N))[0])
    for P in sorted(found):
        n, = found[P]
        rows.append([q, factored(Jc), P, valuation(Jc, P), n, len(levels)])
Table(rows, header=["q", "J_c", "P", "v_P(J_c)", "n_opt", "levels N with T not dividing N"])

q,J_c,P,v_P(J_c),n_opt,levels N with T not dividing N
2,T * (T + 1)^3,T,1,4/3,3
2,T * (T + 1)^3,T + 1,3,3,3
3,2 * T * (T + 1)^4 * (T + 2)^4,T,1,5/4,8
3,2 * T * (T + 1)^4 * (T + 2)^4,T + 1,4,4,8
3,2 * T * (T + 1)^4 * (T + 2)^4,T + 2,4,4,8


## 6. Checks against the literature

Three comparisons with results obtained by entirely different means:

- the closed form of **Bassa–Beelen** for $\Phi_T$ (conjectured by Schweizer), for $q = 2, 3, 4, 5, 9$;
- two specializations printed by **Caranay–Greenberg–Scheidler** (Example 6.2): $\Phi_{T+1}(X, T^2)$ and $\Phi_{T+2}(X, j)$ modulo $T^5 + 2T + 1$, for $q = 3$;
- the identity $\mathrm{Res}_Z\big(\Phi_T(X,Z),\, \Phi_{T+1}(Z,Y)\big) = \Phi_{T(T+1)}(X,Y)$ for $q = 2$, where the three polynomials were computed separately ($\Phi_{T(T+1)}$ at a place of degree 2, since both primes of degree one divide the level).

In [17]:
for N, Phi, _ in data:
    A = N.parent()
    if N == A.gen():
        assert Phi == bassa_beelen_phi_T(A)
        print("q = %d: Phi_T equals the closed form of Bassa-Beelen" % A.base_ring().cardinality())

A.<T> = GF(3)[]
P5 = T^5 + 2*T + 1
assert specialization(load_level(T + 1)[0], 0, T^2, P5) == \
    [T^2, 2*T^3 + 2*T^2 + 2*T + 1, 2*T^4 + 2*T^3 + 2*T^2, 2*T^4 + T^3 + 2*T + 1, 1]
assert specialization(load_level(T + 2)[0], 0, T^4 + T^3 + T^2 + 2*T + 2, P5) == \
    [T^4 + T^2 + 2*T, 2*T^4 + T^3 + 2*T^2 + T, 2*T^4 + T + 2, T^4 + 2*T^3 + 2*T^2 + 1, 1]
print("q = 3: both examples of Caranay-Greenberg-Scheidler are reproduced")

A.<T> = GF(2)[]
R.<t, x, y, z> = GF(2)[]
flat = lambda Phi, u, v: sum(c(t) * u^m * v^n for (m, n), c in coefficient_dict(Phi).items())
PT, PT1, PTT1 = [load_level(N)[0] for N in (T, T + 1, T^2 + T)]
assert flat(PT, x, z).resultant(flat(PT1, z, y), z) == flat(PTT1, x, y)
print("q = 2: Res_Z(Phi_T(X,Z), Phi_(T+1)(Z,Y)) = Phi_(T(T+1))(X,Y)")

q = 2: Phi_T equals the closed form of Bassa-Beelen
q = 3: Phi_T equals the closed form of Bassa-Beelen
q = 4: Phi_T equals the closed form of Bassa-Beelen
q = 5: Phi_T equals the closed form of Bassa-Beelen
q = 9: Phi_T equals the closed form of Bassa-Beelen
q = 3: both examples of Caranay-Greenberg-Scheidler are reproduced
q = 2: Res_Z(Phi_T(X,Z), Phi_(T+1)(Z,Y)) = Phi_(T(T+1))(X,Y)


## 7. Your own computations

Any prime power $q$ and any monic level work. Here is $\Phi_T$ for $q = 4$, where $\mathbb{F}_4 = \mathbb{F}_2(y)$, with the factor predicted by Theorem 1.2(1) at the three primes $T+1$, $T+y$, $T+y+1$ in bold. The lowest coefficient is $J_c^{\,5}$, as Proposition 4.5 says.

In [18]:
A.<T> = GF(4, 'y')[]
Phi = drinfeld_modular_polynomial(T)
primes = [P for P in monic_primes(A, 1) if P != T]
CoefficientTable(Phi, bounds={P: (psi(T), 5) for P in primes})

a_(0,0) = [(T + 1)^25 * (T + y)^25 * (T + y + 1)^25] * T^5
a_(1,0) = [(T + 1)^20 * (T + y)^20 * (T + y + 1)^20] * T^4
a_(1,1) = [(T + 1)^15 * (T + y)^15 * (T + y + 1)^15] * (T^6 + T^3 + y) * (T^6 + T^3 + y + 1)
a_(2,1) = [(T + 1)^10 * (T + y)^10 * (T + y + 1)^10] * T^11
a_(2,2) = [(T + 1)^5 * (T + y)^5 * (T + y + 1)^5] * T^7 * (T^3 + (y + 1)*T + 1) * (T^3 + T + 1) * (T^3 + y) * (T^3 + y + 1) * (T^3 + y*T + 1)
a_(3,1) = [(T + 1)^5 * (T + y)^5 * (T + y + 1)^5] * T^10
a_(3,2) = T^9 * (T + 1) * (T + y) * (T + y + 1) * (T^3 + y) * (T^3 + y + 1)
a_(3,3) = T^2 * (T^3 + (y + 1)*T + 1) * (T^3 + T + 1) * (T^3 + y) * (T^3 + y + 1) * (T^3 + y*T + 1)
a_(4,0) = [(T + 1)^5 * (T + y)^5 * (T + y + 1)^5] * T
a_(4,1) = T^9
a_(4,2) = T^5
a_(4,3) = T
a_(4,4) = 1
a_(5,0) = 1
zero: a_(2,0), a_(3,0), a_(5,1), a_(5,2), a_(5,3), a_(5,4), a_(5,5)

The univariate $\Phi_N(X, 0)$ is much cheaper than the full polynomial, since it needs no deformations and no interpolation. A level of degree 4 takes a few seconds:

In [19]:
A.<T> = GF(2)[]
N = T^4 + T^3 + 1
f, cert = drinfeld_modular_polynomial_at_zero(N, certificate=True)
print("Phi_N(X, 0) has degree %d = psi(N); computed at the place %s with precision K = %d"
      % (f.degree(), cert["place"], cert["precision"]))
print("C_0(N, 0) =", order(f), "  and   C_0(N, T^3 + T^2 + 1) =", order(f, T^3 + T^2 + 1))
assert f == load_level(N, at_zero=True)[0]

Phi_N(X, 0) has degree 17 = psi(N); computed at the place T + 1 with precision K = 420
C_0(N, 0) = 2   and   C_0(N, T^3 + T^2 + 1) = 5


Larger cases take longer: the cost grows roughly like $\psi(N)^3$ multiplications of power series of length $K$. Use `verbose=True` to follow the progress, and see the README for timings. From a shell,

```
sage examples/compute.py 3 "T^2+1"
```

prints a table like the ones above for any $q$ and $N$.

To verify every numerical claim of the paper, or to recompute the data set from scratch:

```
sage paper/verify_paper.py
sage paper/recompute.py all
```